# Twinity/trail-lab 
## Rapport de course


    
    
    
> **NE PAS DIFFUSER - EN COURS DE DEVELOPPEMENT**


Ce notebook est le point d'entrée unique pour analyser une course trail et
comparer à d'autres courses. Il est structuré en quatre sections indépendantes :

| Section |Contenu |
|---|---|
| **0. Configuration** |  Paramètres athlète, courses, flags d'affichage |
| **1. Dernière course** | Carte, profils, dashboard, splits, météo |
| **2. Comparaison multi-courses** |  KPI, profils normalisés, déclin |
| **3. Axes d'amélioration** | Découplage, variabilité, pente, circadien |
| **4. Comparaison deux coureurs** | Profils, delta, radar KPI |

Seule la **Section 0** est à modifier. Les autres s'exécutent automatiquement.

---

> Notebook        : Analyse d'une séance de trail/course    
> Auteur          : Grégory Sainton <trinity4trail@proton.me>    
> Dépôt           : https://github.com/gregs1t/trail    
> Date de création: 2026-05    
> Dernière modif. : 2026-05    
> Version         : 1.0    
> *Licence* : CC BY-NC-SA 4.0    

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from trail_analysis_v2 import (
    # I/O & pipeline
    load_fit, clean_df, load_and_process_race, compute_race_kpis,
    extract_gps_centroid,
    # Terrain
    compute_slope, compute_dplus_dminus, compute_gap,
    segment_updown, classify_walk_run,
    # Physiologie
    compute_hr_zones, compute_cardiac_drift, compute_aerobic_decoupling,
    compute_pace_variability, compute_pace_split, compute_stride_metrics,
    compute_circadian_profile, detect_hitting_wall,  plot_raw_profiles,
    compute_session_load, compute_epoc, plot_epoc,
    # Sections
    section_stats, compute_ravito_stops,
    # Météo
    fetch_weather_hourly, enrich_df_with_weather,
    # Carte
    plot_map,
    # Figures Plotly
    DASHBOARD_VARIABLES,
    plot_profil_colore, plot_dashboard,
    plot_pace_split, plot_hitting_wall,
    plot_aerobic_decoupling, plot_pace_variability,
    plot_walk_by_slope_sections, plot_heatmap_sections,
    plot_circadian_profile, plot_radar_sections,
    plot_weather_along_race, plot_speed_by_slope,
    # Multi-courses
    build_races_table, normalize_by_distance_pct,
    plot_races_comparison, plot_normalized_profiles,
    plot_decay_model, plot_pace_vs_slope_overlay,
    plot_pace_vs_slope_deviation,
    # Comparaison deux coureurs
    plot_comparison_two_runners, build_comparison_table,
    plot_altitude_vs_time,
)

## ⚙️ Section 0 — Configuration

**Seule cette section est à modifier.**

### 0a. Paramètres athlète

In [ ]:
AUTHOR = "Greg."
# # ── ID ──────────────────────────────────────────────────────────────
# NAME = "Laura"
# AGE = 36
# SEX = "Femme"
# # ── Physiologie ──────────────────────────────────────────────────────────────
# FC_MAX    = 200       # Fréquence cardiaque maximale (bpm)
# FC_MIN    = 40        # Fréquence cardiaque de repos (bpm)
# POIDS_KG  = 52.0      # Masse corporelle (kg)

# ── ID ──────────────────────────────────────────────────────────────
NAME = "Greg"
AGE = 49
SEX = "Homme"
# ── Physiologie ──────────────────────────────────────────────────────────────
FC_MAX    = 183       # Fréquence cardiaque maximale (bpm)
FC_MIN    = 47        # Fréquence cardiaque de repos (bpm)
POIDS_KG  = 78.0      # Masse corporelle (kg)



# ── Seuils de traitement ─────────────────────────────────────────────────────
WINDOW_M      = 150.0  # Fenêtre de calcul de la pente (m)
UP_THR        = 3.0    # Seuil montée (%)
DOWN_THR      = -3.0   # Seuil descente (%)
MIN_SEG_M     = 200.0  # Longueur minimale d'un segment (m)
WALK_THR_KMH  = 6.0    # Seuil vitesse marche (km/h)
WALK_THR_CAD  = 140.0  # Seuil cadence marche (spm)

# ── Classes de pente ─────────────────────────────────────────────────────────
SLOPE_BINS   = [-30, -10, -3, 3, 10, 20, 40]
SLOPE_LABELS = ["< -10%", "-10/-3%", "plat", "+3/+10%", "+10/+20%", "> +20%"]



print("Rapport de course - Trail Lab/Twinity")
print(f"Date : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Auteur du rapport : {AUTHOR}")
print("=====================================")
print(f"\nRunner : {NAME}, {AGE} ans\n")
print("=====================================")
print("\nParamètres physiologiques :")
print(f"  {'Sexe':<10} : {SEX}")
print(f"  {'FCmax':<10} : {FC_MAX} bpm")
print(f"  {'FCmin':<10} : {FC_MIN} bpm")
print(f"  {'Poids':<10} : {POIDS_KG} kg")
print("\n")
print("=====================================")
print("\nSeuils de traitement :")
print(f"  - Fenêtre de calcul : {WINDOW_M} m")
print(f"  - Seuil montée : {UP_THR} %")
print(f"  - Seuil descente : {DOWN_THR} %")
print(f"  - Longueur minimale : {MIN_SEG_M} m")
print(f"  - Seuil vitesse marche : {WALK_THR_KMH} km/h")
print(f"  - Seuil cadence marche : {WALK_THR_CAD} spm")


### 0b. Dernière course à analyser

In [ ]:
TARGET = {
    'label':      'SaintéLyon 2025',
    'fit_path':   '/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/SainteLyon_2025-11-29 23:31:00.fit',
    'ravito_km':  [19.2, 34.0, 45.0, 58.8, 65.4],
    'ravito_nom': ['St Christo', 'Ste Catherine', 'St Genou', 'Soucieu', 'Chaponost'],
    # Météo — laisser None pour extraction automatique depuis le FIT
    "lat":        None,
    "lon":        None,
    "date":       "2025-11-29",   # YYYY-MM-DD
    "date_end":   None,           # Pour les courses franchissant minuit
}


print(f"Analyse de la course : {TARGET['label']}")
print(f"Fichier FIT : {TARGET['fit_path']}")
print(f"Date de la course : {TARGET['date']}")
print("Ravitaillements :") 
for i in range(len(TARGET['ravito_km'])):
    print(f"  - {TARGET['ravito_km'][i]} km - {TARGET['ravito_nom'][i]}")


### 0c. Historique multi-courses

Laisser la liste vide `[]` pour désactiver la comparaison.

In [ ]:
# RACES_CATALOG = [
#     {
#          'label':      'Ecotrail 2026',
#          'fit_path':   '/home/gsainton/DATA/Trail/DATA/Laura/22257544267_ACTIVITY_ecotrail_LB.fit',
#          'ravito_km':  [],
#          'ravito_nom': [],
#     },

#     {
#          'label':      'Trail 25/04/2026',
#          'fit_path':   '/home/gsainton/DATA/Trail/DATA/Laura/Running_2026-04-25T16_29_10_LB.fit',
#          'ravito_km':  [],
#          'ravito_nom': [],
#     },
#     {
#          'label':      'Haze 2026',
#          'fit_path':   '/home/gsainton/DATA/Trail/DATA/Laura/TrailRunning_2026-05-03T09_00_02_LB.fit',
#          'ravito_km':  [10.4],
#          'ravito_nom': ['ravito1'],
#     },
# ]

RACES_CATALOG = [
    {
        'label':      'SaintéLyon 2025',
        'fit_path':   '/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/SainteLyon_2025-11-29 23:31:00.fit',
        'ravito_km':  [19.2, 34.0, 45.0, 58.8, 65.4],
        'ravito_nom': ['St Christo', 'Ste Catherine', 'St Genou', 'Soucieu', 'Chaponost'],
    },

    {
         'label':      'Ecotrail80 2026',
         'fit_path':   '/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/EcoTrail de Paris_2026-03-21 10:55:00.fit',
         'ravito_km':  [25.4, 37.7, 47.7, 53.8, 78.3],
         'ravito_nom': ['Buc', 'Chaville', 'Saint Philippe', 'Marcel Bec', 'Saint Cloud'],
    },
    {
         'label':      'Impérial trail 2025',
         'fit_path':   '/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/Imperial trail de Fontainebleau_2025-09-13 06:01:00.fit',
         'ravito_km':  [11, 23.5, 33.5, 46.7, 61],
         'ravito_nom': ['ravito1', 'ravito2', 'ravito3', 'ravito4', 'ravito5'],
    },
    {
         'label':      'Trail de la Haze 2026',
         'fit_path':   '/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/Trail-de-la-haze_2026_05_03 09:00:00.fit',
         'ravito_km':  [10.4],
         'ravito_nom': ['ravito1'],
    },
]




print("\nCatalogue des courses à comparer :")
for race in RACES_CATALOG:
    print(f"  - {race['label']}")


### 0d. Sections à afficher
Zone pour faire apparaitre ou masquer des plots dans le rapport d'analyse.
Mettre `False` pour masquer.


In [ ]:
# ── Section 1 : Dernière course ───────────────────────────────────────────────
SHOW_MAP            = True   # Carte interactive Folium
SHOW_DASHBOARD      = True   # Dashboard FC / allure / dénivelé
SHOW_METEO          = True   # Météo ERA5-Land (nécessite une connexion)
SHOW_SPLITS         = True   # Splits par section (ravitos)
SHOW_PACE_SPLIT     = True   # Analyse positive/négative split
SHOW_HITTING_WALL   = True   # Détection dégradation d'allure
SHOW_HR_ZONES       = True   # Zones de fréquence cardiaque
SHOW_WALK_SLOPE     = True   # % marche par classe de pente
SHOW_HEATMAP        = True   # Heatmap FC (vitesse × pente)
SHOW_SPEED_SLOPE    = True   # Vitesse moyenne par classe de pente
SHOW_RADAR          = True   # Radar par section

# ── Section 2 : Multi-courses ─────────────────────────────────────────────────
SHOW_MULTI_RACE     = True   # Tableau KPI + évolution temporelle
SHOW_NORM_PROFILES  = True   # Profils normalisés 0–100%
SHOW_DECAY          = True   # Modèle de déclin individuel
SHOW_PACE_VS_SLOPE  = True   # Allure vs pente (Minetti)

# ── Section 3 : Axes d'amélioration ───────────────────────────────────────────
SHOW_DECOUPLING     = True   # Découplage aérobie
SHOW_VARIABILITY    = True   # Variabilité d'allure
SHOW_CIRCADIAN      = True   # Profil circadien (courses nocturnes)
SHOW_STRIDE         = True   # Métriques de foulée
#  ── Section 4 : Comparaison ──────────────────────────────────────────────────
SHOW_COMPARISON     = False   # Comparaison avec un autre coureur

---
## 🏃 Section 1 — Dernière course

### 1.0 Chargement et traitement


In [ ]:
df_raw = load_fit(TARGET["fit_path"])
df, ALT_COL = clean_df(df_raw)
df["slope_pct"]  = compute_slope(df, WINDOW_M)
df = compute_gap(df)
df = segment_updown(df, UP_THR, DOWN_THR, MIN_SEG_M)

if "cadence" in df.columns:
    df = classify_walk_run(df, WALK_THR_KMH, WALK_THR_CAD)
if "heart_rate" in df.columns:
    df = compute_cardiac_drift(df, min_speed_kmh=3.0)

dplus, dminus = compute_dplus_dminus(df)
RAVITO_KM  = TARGET["ravito_km"]
RAVITO_NOM = TARGET["ravito_nom"]

dist_km    = df["dist_m"].max() / 1000.0
dur_h      = df["time_h"].max()
start_time = df["timestamp"].iloc[0]

print(f"Course       : {TARGET['label']}")
print(f"Départ       : {start_time.strftime('%Y-%m-%d %H:%M')}")
print(f"Distance     : {dist_km:.1f} km")
print(f"D+           : {dplus:.0f} m  /  D- : {dminus:.0f} m")
print(f"Durée        : {int(dur_h)}h {int((dur_h % 1)*60):02d}min")
print(f"Points GPS   : {len(df)}")

### 1.1 Profils physiologiques bruts

Vue d'ensemble des indicateurs enregistrés par la montre.
Choisir les variables et l'axe temporel, puis exécuter la cellule suivante.
Courbe transparente = données brutes · Courbe pleine = lissage sur `SMOOTHING_S` secondes.

In [ ]:
SMOOTHING_S = 30   # fenêtre de lissage en secondes

_raw_vars = {
    "heart_rate":    "FC (bpm)",
    "pace_s_per_km": "Allure (s/km)",
    "gap_s_per_km":  "GAP (s/km)",
    "speed_kmh":     "Vitesse (km/h)",
    "cadence":       "Cadence (spm)",
    "alt_m":         "Altitude (m)",
    "slope_pct":     "Pente (%)",
    "temperature":   "Température montre (°C)",
    "temp_api":      "Température ERA5 (°C)",
    "power":         "Puissance (W)",
}

_defaults_raw = {"heart_rate", "pace_s_per_km", "alt_m"}
_raw_checkboxes = {
    k: widgets.Checkbox(
        value=(k in _defaults_raw and k in df.columns),
        description=label,
        disabled=(k not in df.columns),
        layout=widgets.Layout(width="280px"),
    )
    for k, label in _raw_vars.items()
}

_raw_toggle = widgets.ToggleButtons(
    options=[("Distance (km)", "distance"), ("Temps (min)", "time")],
    value="distance",
    description="Axe x :",
    style={"button_width": "140px"},
)

_raw_box = widgets.VBox([
    widgets.HTML("<b>Variables à afficher :</b>"),
    *_raw_checkboxes.values(),
    widgets.HTML("<br><b>Axe x :</b>"),
    _raw_toggle,
], layout=widgets.Layout(
    border="1px solid #444", padding="10px",
    border_radius="6px", width="340px",
))
display(_raw_box)

In [ ]:
_selected_raw = [k for k, cb in _raw_checkboxes.items() if cb.value]

if not _selected_raw:
    print("⚠️  Aucune variable sélectionnée.")
else:
    fig = plot_raw_profiles(
        df,
        #variables   = _selected_raw,
        variables   = ["heart_rate", "gap_s_per_km"],
        smoothing_s = SMOOTHING_S,
        x_axis      = _raw_toggle.value,
        ravito_km   = RAVITO_KM,
        ravito_nom  = RAVITO_NOM,
    )
    fig.show()

In [ ]:
if not _selected_raw:
    print("⚠️  Aucune variable sélectionnée.")
else:
    fig = plot_raw_profiles(
        df,
        #variables   = _selected_raw,
        variables   = ["alt_m", "slope_pct"],
        smoothing_s = SMOOTHING_S,
        x_axis      = _raw_toggle.value,
        ravito_km   = RAVITO_KM,
        ravito_nom  = RAVITO_NOM,
    )
    fig.show()

### 1.1 Météo ERA5-Land

Les données météo sont récupérées depuis [Open-Meteo Historical Weather](https://open-meteo.com/)
via le SDK `openmeteo-requests`. Si `lat`/`lon` sont `None` dans TARGET,
les coordonnées sont extraites automatiquement depuis le fichier FIT.

*Référence : Muñoz Sabater J (2019). ERA5-Land hourly data. ECMWF.*

Si vous voulez des détails sur la récupération de la météo : 
[Article blog](https://gregs1t.github.io/trail/2026-04-05-meteo-course-trail/)


In [ ]:
if SHOW_METEO and TARGET.get("date"):
    lat = TARGET.get("lat")
    lon = TARGET.get("lon")
    if lat is None or lon is None:
        lat, lon = extract_gps_centroid(df)
        if lat is not None:
            print(f"ℹ️  Coordonnées extraites du FIT : {lat:.4f}, {lon:.4f}")
        else:
            print("⚠️  Pas de coordonnées GPS — météo ignorée.")

    if lat is not None:
        df_weather = fetch_weather_hourly(
            lat      = lat,
            lon      = lon,
            date_str = TARGET["date"],
            date_end = TARGET.get("date_end"),
        )
        if df_weather is not None:
            df = enrich_df_with_weather(df, df_weather)
            print("✅ Météo ERA5-Land intégrée.")
        else:
            print("⚠️  Météo indisponible — vérifier la connexion.")
else:
    print("ℹ️  Météo désactivée (SHOW_METEO=False ou date manquante).")

### 1.2 Carte GPS

In [ ]:
if SHOW_MAP:
    m = plot_map(
        df,
        ravito_km     = RAVITO_KM,
        ravito_nom    = RAVITO_NOM,
        color_col     = "heart_rate",
        animated      = True,
        show_elevation= False,
    )
    display(m)

### 1.3 Dashboard physiologique

Profil altimétrique coloré par les variables physiologiques sélectionnées.
Cocher les panneaux souhaités, puis exécuter la cellule suivante.


In [ ]:
_all_vars = {
    "heart_rate":    "FC (bpm)",
    "pace_s_per_km": "Allure (s/km)",
    "gap_s_per_km":  "GAP (s/km)",
    "temperature":   "Température montre (°C)",
    "temp_api":      "Température ERA5-Land (°C)",
    "speed_kmh":     "Vitesse (km/h)",
    "cadence":       "Cadence (spm)",
    "slope_pct":     "Pente (%)",
}

_defaults = {"heart_rate", "pace_s_per_km", "gap_s_per_km"}
_checkboxes = {
    k: widgets.Checkbox(
        value=(k in _defaults and k in df.columns),
        description=label,
        disabled=(k not in df.columns),
        layout=widgets.Layout(width="280px"),
    )
    for k, label in _all_vars.items()
}

_box = widgets.VBox(
    [widgets.HTML("<b>Panneaux à afficher :</b>")] + list(_checkboxes.values()),
    layout=widgets.Layout(
        border="1px solid #444", padding="10px",
        border_radius="6px", width="320px",
    ),
)
display(_box)

In [ ]:
if SHOW_DASHBOARD:
    selected_keys = [k for k, cb in _checkboxes.items() if cb.value]
    if not selected_keys:
        print("⚠️  Aucun panneau sélectionné.")
    else:
        variables = {
            k: DASHBOARD_VARIABLES.get(k, (k, "Viridis", None, None))
            for k in selected_keys if k in df.columns
        }
        if "heart_rate" in variables:
            lbl, cs, _, _ = variables["heart_rate"]
            variables["heart_rate"] = (lbl, cs, FC_MIN, FC_MAX)
        fig = plot_dashboard(
            df, FC_MIN, FC_MAX,
            variables  = variables,
            ravito_km  = RAVITO_KM,
            ravito_nom = RAVITO_NOM,
        )
        fig.show()

In [ ]:
# ── Charge de séance : TRIMP, EPOC, PTE, Calories ────────────────────────────
AGE_YEARS = 35       # âge du coureur (pour Keytel)
VO2MAX    = None     # VO2max en ml/kg/min — None = 50 par défaut

load_result = compute_session_load(
    df,
    fc_max    = FC_MAX,
    fc_rest   = FC_MIN,
    poids_kg  = POIDS_KG,
    vo2max    = VO2MAX,
    age_years = AGE_YEARS,
    gender    = "male",       # "male" | "female"
)

# Affichage formaté
w = 28
# Charge de séance
load_result = compute_session_load(
    df,
    fc_max    = FC_MAX,
    fc_rest   = FC_MIN,
    poids_kg  = POIDS_KG,
    vo2max    = VO2MAX,
    age_years = AGE_YEARS,
    gender    = "male",
)

print("\nCharge de séance")
print(f"  {'─'*50}")
print(f"  {'TRIMP (Banister)':<{w}} : {load_result['trimp']:.1f} AU")
print(f"  {'EPOC peak':<{w}} : {load_result['epoc_peak_ml_kg']:.1f} ml/kg")
print(f"  {'PTE (0–5)':<{w}} : {load_result['pte']:.1f}")
print(f"  {'─'*50}")
print(f"  {'Calories (Keytel FC)':<{w}} : "
      f"{load_result['calories_keytel'] or '—'} kcal")
print(f"  {'Calories (distance+D+)':<{w}} : "
      f"{load_result['calories_mechanical']} kcal")
print(f"  {'Calories combinées':<{w}} : "
      f"{load_result['calories_combined']} kcal")

fig = plot_epoc(df, FC_MAX, vo2max=VO2MAX,
                ravito_km=RAVITO_KM, ravito_nom=RAVITO_NOM)
fig.show()

### Charge de séance — TRIMP, EPOC, PTE et Calories



#### TRIMP — Training Impulse

Le TRIMP (Banister 1975) est la mesure de charge d'entraînement la plus
utilisée en sciences du sport. Il combine la durée de l'effort et l'intensité
cardiaque relative (FC réserve) avec une pondération exponentielle qui
reflète la non-linéarité de la relation intensité/fatigue.

$$\text{TRIMP} = \sum \Delta t \times HRr \times 0.64 \times e^{b \cdot HRr}$$

où $HRr = (FC - FC_{repos}) / (FC_{max} - FC_{repos})$ et $b = 1.92$ (homme)
ou $1.67$ (femme). Un TRIMP de 100–150 correspond typiquement à une longue
sortie d'endurance, > 200 à une course exigeante.



#### EPOC — Excess Post-Exercise Oxygen Consumption

L'EPOC mesure la perturbation de l'homéostase causée par la séance — la
quantité d'oxygène consommée en excès pendant la récupération. C'est
l'indicateur physiologique de la "dette" accumulée pendant l'effort.

Le modèle implémenté ici suit la structure récursive publiée par Firstbeat
(Saalasti 2003 / Rusko et al. 2003) : l'EPOC s'accumule quand l'intensité
dépasse ~50 % du VO2max et se dissipe exponentiellement pendant les phases
de récupération. La courbe EPOC au fil de la course permet de voir à quel
moment la fatigue s'accumule réellement.

> ⚠️ L'algorithme exact de Firstbeat (utilisé dans les montres Suunto et
> Garmin) est **propriétaire** — les coefficients précis ne sont pas publiés.
> L'implémentation ci-dessus est une approximation basée sur le white paper
> public, avec une erreur typique de ±15–20 ml/kg par rapport aux valeurs
> Suunto. Les valeurs absolues ne sont donc **pas directement comparables**
> à celles de ta montre, mais les tendances et comparaisons entre séances
> restent valides.



#### PTE — Peak Training Effect

Le PTE normalise l'EPOC peak par le niveau de fitness de l'athlète (VO2max).
Un athlète plus entraîné a besoin d'un EPOC plus élevé pour atteindre le
même niveau de stimulus. L'échelle 0–5 est celle de Suunto/Firstbeat :

| PTE | Effet |
|---|---|
| 1.0–1.9 | Récupération active — effet mineur |
| 2.0–2.9 | Maintien de la forme |
| 3.0–3.9 | Amélioration aérobie |
| 4.0–4.9 | Amélioration importante — sollicitation élevée |
| 5.0 | Effort maximal — à ne pas répéter souvent |

> ⚠️ Le PTE dépend fortement du VO2max renseigné dans `VO2MAX`. Si cette
> valeur n'est pas connue, la valeur par défaut de 50 ml/kg/min est utilisée,
> ce qui peut surestimer ou sous-estimer le PTE selon le profil du coureur.


#### Calories

Deux méthodes sont calculées et combinées :

**Keytel et al. (2005)** — basée sur la FC moyenne, le poids, l'âge et le sexe.
Validée sur tapis roulant à intensité sous-maximale. Plus précise sur les
efforts réguliers, moins fiable sur les ultras avec fort dénivelé.

**Heuristique mécanique** — basée sur la distance et le D+ :
$\text{kcal} \approx \text{poids} \times \text{distance} + \text{poids} \times 0.00235 \times D+$

La valeur **combinée** (70 % Keytel + 30 % mécanique) est le meilleur
estimateur sur un trail avec dénivelé.

*Références : Banister EW (1975). Aust J Sports Med 7(3):57-61. /
Saalasti S (2003). PhD thesis, Univ. Jyväskylä. /
Firstbeat (2012). White Paper EPOC. /
Keytel LR et al. (2005). J Sports Sci 23(3):289-297. /
di Prampero PE (1986). Int J Sports Med 7(2):55-72.*

### 1.4 Conditions météo au fil du parcours

#### Le WBGT — pourquoi cet indicateur plutôt que la température seule ?

La température de l'air ne suffit pas à évaluer le stress thermique réel subi par un coureur. Un athlète qui court produit de la chaleur métabolique et doit l'évacuer principalement par la sudation. L'efficacité de cette évaporation dépend de l'humidité ambiante et du rayonnement solaire — deux facteurs que la température seule ignore.

Le **Wet Bulb Globe Temperature** (WBGT) intègre ces trois dimensions :

$$\text{WBGT} \approx 0.567 \cdot T_{air} + 0.393 \cdot e_a + 3.94 + 0.0006 \cdot R_{sol}$$

où $T_{air}$ est la température de l'air (°C), $e_a$ la pression de vapeur d'eau (hPa) dérivée de l'humidité relative, et $R_{sol}$ le rayonnement solaire global (W/m²).

Cette formulation est celle de **Bernard & Kenney (1994)**, adaptée pour le calcul à partir de données de réanalyse ERA5-Land. En conditions réelles, le WBGT se mesure avec trois thermomètres distincts : un thermomètre sec, un thermomètre humide (qui capte l'effet de l'évaporation) et un globe noir (qui capte le rayonnement solaire). La formule ci-dessus reproduit ce résultat à partir des données ERA5-Land, sans nécessiter ces instruments.



#### Seuils de risque en course à pied

Les seuils retenus dans Twinity sont issus de **Périard et al. (2021)** et **Casa et al. (2015)** :

| WBGT | Niveau de risque | Recommandation |
|---|---|---|
| < 18 °C | ✅ Faible | Conditions optimales |
| 18–23 °C | 🟡 Modéré | Hydratation régulière |
| 23–28 °C | 🟠 Élevé | Réduire l'intensité, surveiller les signes de coup de chaleur |
| 28–32 °C | 🔴 Très élevé | Risque sérieux — adapter le rythme significativement |
| > 32 °C | ⛔ Danger | Arrêt recommandé pour les sujets non acclimatés |

Ces seuils sont définis pour des efforts prolongés (> 1 h) chez des athlètes entraînés. Pour un ultra-trail de nuit comme la SaintéLyon, le WBGT reste généralement bas (c'est ce que j'ai pu observer), mais un épisode chaud en début de course peut suffire à déclencher une déshydratation progressive invisible sur le moment.

#### Limites de l'approche ERA5-Land

Le modèle ERA5-Land fournit des données horaires à une résolution spatiale d'environ 9 km. Pour une course en montagne avec de forts gradients altitudinaux, les valeurs météo correspondent au centroïde GPS de la trace et non à chaque point du parcours. L'interpolation temporelle sur le tracé GPS introduit une approximation supplémentaire. Ces données restent néanmoins bien supérieures à l'absence d'information météo, et cohérentes avec les mesures de terrain dans la grande majorité des études de validation (Muñoz Sabater 2019).

#### Références

- Bernard TE & Kenney WL (1994). Rationale for a personal monitor for the wet bulb globe temperature. *AIHAJ* 55(1):32-36.
- Périard JD, Racinais S & Sawka MN (2021). Adaptations and mechanisms of human heat acclimation. *Br J Sports Med* 55(15):865-876.
- Casa DJ et al. (2015). National Athletic Trainers' Association position statement: exertional heat illnesses. *J Athl Train* 50(9):986-1000.
- Muñoz Sabater J (2019). ERA5-Land hourly data from 1981 to present. *Copernicus Climate Change Service (C3S) Climate Data Store*. [doi:10.24381/cds.e2161bac](https://doi.org/10.24381/cds.e2161bac)

In [ ]:
if SHOW_METEO and "temp_api" in df.columns:
    fig = plot_weather_along_race(df, RAVITO_KM, RAVITO_NOM,
                                   show_watch_temp=False)
    fig.show()

### 1.5 Statistiques par section

Tableau récapitulatif des performances entre chaque ravitaillement.


In [ ]:
if SHOW_SPLITS:
    df_splits = section_stats(df, RAVITO_KM, RAVITO_NOM)
    display(df_splits)
    print()
    df_stops = compute_ravito_stops(df, RAVITO_KM, RAVITO_NOM)
    display(df_stops)

### 1.6 Zones de fréquence cardiaque

Distribution du temps par zone cardiaque (%FCmax).


In [ ]:
if SHOW_HR_ZONES and "heart_rate" in df.columns:
    zones_df = compute_hr_zones(df, FC_MAX)
    display(zones_df)

### 1.7 Analyse du split (positif / négatif)

#### Le GAP — Grade Adjusted Pace

En trail, l'allure brute (min/km) est une mesure trompeuse pour évaluer la régularité de l'effort : courir à 8 min/km en montée à 15 % n'a rien à voir physiologiquement avec courir à 8 min/km sur terrain plat. Le **GAP (Grade Adjusted Pace)** corrige ce biais en ramenant chaque segment à son équivalent sur terrain plat.

Le calcul s'appuie sur le modèle de Minetti et al. Le ratio entre le coût sur terrain plat $C(0) \approx 3.6$ J/(kg·m) et le coût à la pente $i$ donne le facteur de correction de vitesse :

$$\text{GAP} = \frac{\text{Allure brute}}{\,C(0)\,/\,C(i)\,}$$

Concrètement : une montée à 10 % multiplie le coût métabolique par environ 2.3 — le GAP sera donc 2.3 fois plus rapide que l'allure brute observée, ce qui la ramène à son équivalent plat. Une descente à −10 % réduit légèrement le coût (optimum autour de −10 % selon Minetti), produisant un GAP légèrement plus lent que l'allure brute.

Le GAP est la métrique utilisée dans **toutes les analyses de ce notebook** dès qu'une comparaison d'effort est nécessaire — splits, découplage, variabilité, déclin. C'est lui qui rend comparables des sections de profils très différents. Ce calcul est également celui utilisé dans la plupart de nos montres de sport (Suunto, Garmin, Coros...)

> **WARNING** : Les limites du modèle Minetti, on en a déjà parlé, elles s'appliquent ici aussi (voir section 2.5) : sur tapis roulant pas de prise en compte de la fatigue, du terrain naturel, ni de la variabilité entre les coureurs. Le GAP est un indicateur d'effort relatif, pas une mesure absolue.

#### Analyse du split positif / négatif

Un **split positif** signifie que la seconde moitié de la course est plus lente que la première (ratio $GAP_2/GAP_1$ > 1). C'est le cas le plus courant, en particulier sur les formats longs. Un **split négatif** (ratio < 1) signifie que le coureur a accéléré sur la seconde moitié — signe d'une gestion très conservatrice au départ ou d'une montée en régime progressive.

Le plot montre deux éléments complémentaires :

**Panel haut — courbe GAP lissée** : l'évolution de l'allure corrigée au fil de la distance. La ligne de référence (tirets gris) est le GAP médian de la première moitié. Les zones vertes indiquent les passages plus rapides que la référence, les zones rouges les passages plus lents. La ligne en pointillets, verticale marque la mi-course.

**Panel bas — delta par section** : l'écart de GAP entre chaque section et la première section de référence, exprimé en pourcentage. Un delta de +20 % sur la dernière section signifie que le coureur était 20 % plus lent (en termes d'effort normalisé) que sur la première section.

In [ ]:
if SHOW_PACE_SPLIT:
    fig = plot_pace_split(df, RAVITO_KM, RAVITO_NOM)
    fig.show()

### 1.8 Détection de dégradation d'allure (*Hitting the Wall*)

#### Qu'est-ce que "Hitting the Wall" ?

Le terme *hitting the wall* désigne un épisode de dégradation brutale et soutenue de l'allure, distinct du ralentissement progressif naturel en fin d'effort. En marathon, il est classiquement associé à l'épuisement des réserves de glycogène aux alentours du km 30–35. En ultra-trail, le mécanisme est plus complexe et multifactoriel : fatigue neuromusculaire, hypoglycémie, déshydratation, dégradation cognitive, ou combinaison de plusieurs de ces facteurs simultanément.

La caractéristique clé qui distingue un épisode *hitting the wall* d'un simple ralentissement passager est sa **durée et son amplitude** : le coureur ne ralentit pas temporairement dans une côte difficile, il ralentit de façon soutenue sur plusieurs kilomètres consécutifs, et ne récupère pas son allure de référence une fois la difficulté passée.




#### Comment est détecté l'épisode ici

L'algorithme suit la méthode proposée par Prigent et al. (2024). La course est d'abord divisée en deux zones :

**Fenêtre de référence** : les premiers kilomètres de la course (de 5 km au quart de la distance par défaut), où le coureur est encore frais et où l'allure reflète ses capacités du jour. Le GAP médian sur cette fenêtre constitue la **baseline individuelle**.

**Seuil de détection** : un épisode est déclenché quand le GAP lissé dépasse la baseline de plus de **+25 %** de façon continue sur plus de **5 km consécutifs**. Ces deux paramètres sont configurables dans la cellule de configuration.

Le GAP est utilisé ici plutôt que l'allure brute pour la même raison qu'en section 1.7 : neutraliser les variations dues au relief et ne détecter que les dégradations liées à l'état du coureur.

#### Comment lire le plot

La **ligne en tirets verts** est le GAP de référence (baseline individuelle). La **ligne en tirets rouges** est le seuil de détection (baseline × 1.25 par défaut). La **courbe bleue** est le GAP lissé sur une fenêtre glissante.

Les **zones rouges en fond** indiquent les épisodes détectés, avec le pourcentage de dégradation affiché en en-tête. La **zone verte** en début de course matérialise la fenêtre de référence. Si le profil altimétrique est activé, il apparaît en arrière-plan sur l'axe secondaire droit — il permet de vérifier immédiatement si un épisode coïncide avec une section difficile du parcours ou s'il se produit sur du terrain plus favorable.

Un épisode détecté sur une montée raide est moins préoccupant qu'un épisode détecté sur une section plate ou une descente : dans le premier cas, le GAP est mécaniquement plus difficile à maintenir ; dans le second, la dégradation est plus probablement liée à l'état du coureur.



#### Ce que ça apporte

La localisation des épisodes sur le parcours est l'information la plus utile. Trois patterns sont fréquents :

**Pattern 1 — Épisode unique en fin de course (> 70 %)** : le plus courant sur les ultras bien gérés. Le coureur a tenu son allure jusqu'à épuisement progressif des ressources. Axes de travail : volume de préparation, protocole de nutrition en course, travail de résistance à la fatigue.

**Pattern 2 — Épisode précoce (entre 30 et 60 %)** : signal d'un problème de gestion (départ trop rapide) ou d'un événement exogène (problème digestif, chute, déshydratation). À croiser avec le profil de découplage (section 3.1) et les notes de course pour identifier la cause.

**Pattern 3 — Épisodes multiples et répétés** : le coureur oscille entre des phases de récupération partielle et des rechutes. Pattern caractéristique d'une gestion en dents de scie — accélérations trop importantes aux ravitaillements ou dans les descentes, suivies de défaillances en montée. Axe de travail : discipline de gestion de l'effort, conscience de l'allure cible.

#### Calibration des paramètres

Les valeurs par défaut (+25 %, 5 km) sont issues de Prigent et al. (2024) et adaptées au contexte trail. Pour un coureur très régulier, abaisser le seuil à +15 % permet de détecter des épisodes plus subtils. Pour une course avec beaucoup de dénivelé et une variabilité naturellement élevée, remonter à +30 % évite les faux positifs sur les sections difficiles.

*Références : Prigent G, Doyle-Baker D, Rustarazo-Valverde M, Bini RR & Cardinale M (2024). Identification of performance deterioration episodes during ultra-trail running. Front Physiol 15. PMC12575221. / Buman MP, Brewer BW, Cornelius AE, Van Raalte JL & Petitpas AJ (2008). Hitting the wall in the marathon. Psychol Sport Exerc 9(2):177-190.*

In [ ]:
if SHOW_HITTING_WALL:
    fig = plot_hitting_wall(
        df, RAVITO_KM, RAVITO_NOM,
        threshold_pct   = 25.0,
        min_duration_km = 5.0,
        ref_start_km    = 5.0,
        show_elevation  = True,
    )
    fig.show()

### 1.9 Probabilité de marcher par classe de pente

In [ ]:
if SHOW_WALK_SLOPE and "cadence" in df.columns:
    fig = plot_walk_by_slope_sections(
        df, SLOPE_BINS, SLOPE_LABELS,
        ravito_km = RAVITO_KM,
    )
    fig.show()

### 1.10 Heatmap FC médiane (vitesse × pente)



#### Ce que représente cette heatmap
Chaque cellule de la grille correspond à une combinaison vitesse/pente et affiche la FC médiane observée à cet endroit. La couleur va du vert (FC basse, effort modéré) au rouge (FC élevée, effort intense). Les cellules grises n'ont pas assez de points pour être fiables (minimum 5 points par cellule).
La ligne verticale tiretée noire marque le seuil marche/course (~6 km/h). La ligne horizontale tiretée noire marque la pente 0 %.
Comment lire la heatmap globale (Course entière)
La heatmap globale donne le profil cardiaque caractéristique du coureur sur ce type de terrain. Un coureur bien entraîné en trail montrera :

- Une FC modérée en descente même à vitesse élevée — signe d'une bonne économie de descente
- Une FC élevée en montée raide à faible vitesse — normal, c'est le coût du dénivelé
- Une plage de FC relativement homogène sur terrain plat à allure de course — signe de régularité aérobie

#### Comment lire les heatmaps par section
La comparaison section par section est l'information la plus riche pour un coach. Trois questions à se poser :

**La FC monte-t-elle globalement d'une section à l'autre ?**
Si les mêmes combinaisons vitesse/pente produisent une FC de plus en plus élevée, c'est le signal d'une fatigue cardiovasculaire progressive — confirmé par le découplage (section 3.1).

**La distribution des points se déplace-t-elle vers la gauche (vitesses plus basses) ?**
Si le coureur ralentit sur les mêmes classes de pente en fin de course, les points se concentrent sur des vitesses inférieures. La heatmap devient "vide" sur la droite.

**La FC en descente augmente-t-elle anormalement en fin de course ?**
Une FC élevée en descente (pentes négatives) est rare en début de course mais peut apparaître en fin d'ultra — signe d'une fatigue neuromusculaire importante qui force le recrutement de fibres supplémentaires pour contrôler la descente.

In [ ]:
if SHOW_HEATMAP and "heart_rate" in df.columns:
    fig = plot_heatmap_sections(df, FC_MAX, ravito_km=RAVITO_KM)
    fig.show()

### 1.11 Vitesse moyenne par classe de pente

Vitesse moyenne ± écart-type par classe de pente. Identifie les classes
où le coureur perd le plus de temps — souvent révélateur d'un manque
technique en montée ou d'une prudence excessive en descente.


#### Ce que représente ce plot
Chaque barre représente la vitesse moyenne sur une classe de pente, avec les barres d'erreur indiquant l'écart-type — c'est-à-dire la dispersion des vitesses autour de la moyenne sur cette classe. Si des sections sont activées (via ravito_km), les barres sont groupées par section.

#### Comment lire les barres d'erreur
Une barre d'erreur courte signifie que le coureur est très régulier sur cette classe de pente — il gère cet angle de façon reproductible à chaque fois qu'il le rencontre.
Une barre d'erreur longue signifie que la vitesse est très variable sur cette classe — cela peut refléter des conditions de terrain hétérogènes (certaines sections de cette pente sont techniques, d'autres non), ou une gestion irrégulière de l'effort.

#### Lecture par classe de pente

- Terrain plat (−3 % à +3 %) : c'est la vitesse de référence du coureur. L'écart-type ici reflète la variabilité globale de gestion d'allure sur terrain neutre.
- Montées modérées (+3 % à +10 %) : zone de transition course/marche. Un écart-type élevé dans cette zone indique que le coureur alterne course et marche sans stratégie claire — à croiser avec la section 1.9 (probabilité de marcher par classe de pente).
- Montées raides (> +10 %) : la marche est souvent dominante. La vitesse est basse mais devrait être régulière si le coureur a une technique de montée bien établie.
- Descentes (< −3 %) : une vitesse élevée en descente avec un faible écart-type = coureur technique et confiant. Un écart-type élevé en descente = prudence variable, souvent liée à la fatigue ou à la qualité du terrain.

In [ ]:
if SHOW_SPEED_SLOPE:
    fig = plot_speed_by_slope(
        df,
        slope_bins   = SLOPE_BINS,
        slope_labels = SLOPE_LABELS,
        ravito_km    = RAVITO_KM,
    )
    fig.show()

---
## Comparaison multi-courses

Cette section permet de mettre en regard plusieurs courses d'une même personne. Cela permet de voir d'éventuels effets de l'entrainement au fil d'une saison, d'une année sur l'autre... 


> Nécessite au moins 2 courses dans `RACES_CATALOG`.


In [ ]:
races = []
if RACES_CATALOG:
    for cfg in RACES_CATALOG:
        print(f"Chargement : {cfg['label']} ...", end=" ")
        try:
            race = load_and_process_race(
                fit_path   = cfg["fit_path"],
                fc_max     = cfg.get("fc_max",   FC_MAX),
                fc_min     = cfg.get("fc_min",   FC_MIN),
                poids_kg   = cfg.get("poids_kg", POIDS_KG),
                ravito_km  = cfg["ravito_km"],
                ravito_nom = cfg["ravito_nom"],
            )
            race["meta"]["name"] = cfg["label"]
            races.append(race)
            print("✅")
        except Exception as e:
            print(f"❌ {e}")
    print(f"\n{len(races)} course(s) chargée(s).")
else:
    print("ℹ️  Aucune course dans RACES_CATALOG — Section 2 ignorée.")

### 2.1 Tableau de synthèse des KPI

Une ligne par course. Le GAP neutralise le biais topographique.


In [ ]:
if races and SHOW_MULTI_RACE:
    df_table = build_races_table(races)
    display_cols = [
        "name", "date", "distance_km", "dplus_m", "duration_h",
        "gap_med_s_km", "cv_gap_pct", "split_ratio",
        "fc_frac", "pct_walk", "decoupling_max",
    ]
    display_cols = [c for c in display_cols if c in df_table.columns]
    display(df_table[display_cols])

### 2.2 Évolution temporelle des métriques

Chaque panneau montre l'évolution d'un indicateur au fil des courses.
Une tendance linéaire est ajoutée automatiquement à partir de 3 courses.
Là encore, il faut garder à l'esprit que si la technicité du terrain est différente d'une course à l'autre, les indicateurs ne sont pas forcément comparables. 


| Indicateur | Définition | Tendance favorable |
|---|---|---|
| **GAP médian** | Allure corrigée du dénivelé (Minetti 2002) | ↓ plus rapide |
| **CV GAP** | Coefficient de variation de l'allure — régularité de l'effort | ↓ plus régulier |
| **Ratio split** | GAP 2e moitié / GAP 1re moitié — 1.0 = parfait, >1 = ralentissement | → 1.0 |
| **FC / FC max** | Fraction de la FC maximale — intensité relative | Stable ou ↓ |
| **% marche** | Proportion du temps passé à marcher | Contextuel |
| **Découplage max** | Dérive du ratio FC/GAP — signal de fatigue cardiovasculaire (seuil +2.5 %) | ↓ |


In [ ]:
if races and SHOW_MULTI_RACE and len(races) >= 2:
    fig = plot_races_comparison(
        df_table,
        metrics=["gap_med_s_km", "cv_gap_pct", "split_ratio",
                 "fc_frac", "pct_walk", "decoupling_max"],
    )
    fig.show()

### 2.3 Profils normalisés (0–100 % de la distance)

Les courses sont ramenées sur un axe commun 0–100 %.
Ca permet de comparer des courses de longueurs différentes.
On ajoute aussi une moyenne (ça va sens s'il y a plusieurs courses en fait)

> Si les courses ont des distances trop différentes, je trouve que cette comparaison n'a plus de sens. On ne fourni par le même effort entre un ultra et un 10km et si l'effort en valeur absolu est comparable, il n'est certainement pas réparti de la même manière. Mais je trouvais cette représentation intéressante


*Kerhervé HA et al. (2015). PLoS ONE 10(12):e0145482.*


In [ ]:
if races and SHOW_NORM_PROFILES and len(races) >= 2:
    fig = plot_normalized_profiles(races, col="gap_s_per_km",
                                   show_mean=True, show_ci=True)
    fig.show()


In [ ]:
if races and SHOW_NORM_PROFILES and len(races) >= 2:
    fig = plot_normalized_profiles(races, col="heart_rate",
                                   show_mean=True, show_ci=True)
    fig.show()

### 2.4 Modèle de déclin individuel

#### Qu'est-ce que ce plot représente ?

Ce plot répond à une question simple : **est-ce que je ralentis de façon prévisible au fil de mes courses, et si oui, à partir de quel moment ?**

Le principe est le suivant. Toutes les courses sont ramenées sur un axe commun 0–100 % de la distance (voir section 2.3). Sur cet axe normalisé, on calcule le profil GAP moyen de toutes tes courses, puis on ajuste un polynôme de degré 2 sur ce profil moyen. Ce polynôme est ton **modèle de déclin individuel** : il décrit ta façon "habituelle" de gérer l'effort sur la durée d'une course.

#### Comment lire le panneau gauche

La courbe en tirets blancs  est le modèle polynomial. Les courbes colorées sont tes courses individuelles. Si toutes tes courses suivent à peu près la même trajectoire que le modèle, ton profil d'effort est **reproductible** — c'est une bonne nouvelle, ça signifie que tu te connais bien et que tu paces de façon cohérente. Si une course s'écarte fortement, c'est qu'il s'est passé quelque chose de particulier ce jour-là (météo, fatigue, terrain).

L'axe GAP est inversé : vers le bas = plus rapide. Une courbe qui monte (ralentissement) en fin de course est attendue — l'enjeu est de contrôler **l'amplitude** de ce ralentissement.

#### Comment lire le panneau droit (résidus)

Le résidu est la différence entre ton GAP réel et le GAP prédit par le modèle, point par point sur la distance normalisée.

- **Résidu positif** (au-dessus de zéro) : tu étais plus lent.e que ce que le modèle prédit — fatigue plus importante qu'attendu, mauvaise journée, ou départ trop rapide qui se paye plus tard.
- **Résidu négatif** (en dessous de zéro) : tu étais plus rapide que prévu — bonne forme, conditions favorables, ou départ plus conservateur.

La zone verte ±10 s/km matérialise la marge d'erreur raisonnable. Un résidu qui reste dans cette zone sur toute la course signifie que tu as couru "dans le modèle".

#### Ce que ça apporte

Deux patterns sont particulièrement utiles à identifier :

**Pattern 1 — Résidu croissant en fin de course** : le coureur commence dans le modèle et décroche à partir de 60–70 %. C'est le signe classique d'un départ trop rapide ou d'une insuffisance de préparation spécifique sur la résistance à la fatigue. Axe de travail : sorties longues à allure modérée, travail de fin de course.

**Pattern 2 — Résidu élevé dès le début** : le coureur est systématiquement plus lent que son propre modèle sur une course donnée. Ce n'est pas un problème de gestion — c'est une forme du jour insuffisante ou des conditions extérieures défavorables. Utile pour interpréter une "mauvaise" course sans sur-corriger l'entraînement.

#### Limites du modèle

Le polynôme de degré 2 est une approximation intentionnellement simple. Il capture la tendance globale (déclin progressif) mais ne modélise pas les variations liées au profil topographique — c'est le rôle du GAP, qui neutralise déjà une partie de cet effet. Avec moins de 3 courses, le modèle n'est pas fiable et la courbe moyenne doit être interprétée avec précaution.

*Références : Matta GG et al. (2020). Pacing strategy in trail running. Eur J Sport Sci 20(3):347-356. / Kerhervé HA et al. (2015). The dynamics of running pace adjustment in trail running. PLoS ONE 10(12):e0145482.*

In [ ]:
if races and SHOW_DECAY and len(races) >= 2:
    fig = plot_decay_model(races, col="gap_s_per_km", degree=2)
    fig.show()

### 2.5 Allure vs pente — comparaison au modèle Minetti

#### Rappel du modèle Minetti (2002)

Le modèle Minetti est une courbe du coût métabolique de la course en fonction de la pente, établie en laboratoire sur tapis roulant à des pentes allant de −45 % à +45 % :

$$C(i) = 155.4\,i^5 - 30.4\,i^4 - 43.3\,i^3 + 46.3\,i^2 + 19.5\,i + 3.6$$

où $i$ est la pente en fraction décimale (ex. 0.10 pour 10 %) et $C$ le coût en J/(kg·m). En supposant une puissance métabolique constante, on en déduit une vitesse théorique pour chaque pente à partir de la vitesse de référence sur terrain plat.

Ce modèle est la base du GAP (Grade Adjusted Pace) utilisé dans tout ce notebook.


#### Comment lire le panneau gauche

Chaque point représente la vitesse médiane observée sur une classe de pente, pour chaque course. La courbe noire est la prédiction de Minetti calibrée sur ta vitesse de référence sur terrain plat.

- Un point **au-dessus** de la courbe Minetti : tu vas plus vite que prédit — efficace sur cette classe de pente.
- Un point **en dessous** : tu vas moins vite que prédit — perte d'efficacité sur cette pente.

#### Comment lire le panneau droit (écart relatif à Minetti)

L'écart relatif est $(V_{réelle} - V_{Minetti}) / V_{Minetti}$, exprimé en pourcentage. La zone verte ±5 % représente la marge dans laquelle le modèle est considéré comme bien calibré.

Un écart systématiquement négatif sur les montées raides (+10/+20 %) suggère un manque de puissance musculaire ou de technique de montée. Un écart négatif en descente raide indique souvent une prudence excessive ou une faiblesse excentrique des quadriceps — fréquent chez les coureurs peu habitués aux dénivelés importants.

#### Ce que ça apporte

Ce plot est particulièrement utile pour **cibler les axes techniques** plutôt que les axes physiologiques :

- Retard en montée raide (> +10 %) → travail de montée spécifique, renforcement postérieur, technique de marche rapide avec bâtons
- Retard en descente raide (< −10 %) → renforcement excentrique des quadriceps, sorties descente technique, travail de confiance sur terrain glissant
- Retard sur terrain plat → problème de vitesse de base ou de gestion globale de l'allure, pas lié au profil


*Références : Minetti AE, Moia C, Roi GS, Susta D & Ferretti G (2002). Energy cost of walking and running at extreme uphill and downhill slopes. J Appl Physiol 93(3):1039-1046. / Vernillo G et al. (2017). Biomechanics and physiology of uphill and downhill running. Sports Med 47(4):615-629. / Giandolini M et al. (2016). Impact reduction during running: efficiency of simple acute interventions in recreational runners. Eur J Sport Sci 16(8):1010-1017.*


> ⚠️ **Avertissement sur la validité du modèle Minetti**
>
> Le modèle Minetti a été établi **en laboratoire**, sur tapis roulant, sur des sujets entraînés mais non spécialistes de l'ultra-trail, à des vitesses et des durées d'effort très inférieures à celles d'un ultra. Son application à des courses de plusieurs heures sur terrain naturel introduit plusieurs sources d'erreur importantes :
>
> **1. Fatigue neuromusculaire.** Le coût métabolique augmente avec la fatigue, en particulier en descente où le travail excentrique dégrade progressivement l'économie de course. Le modèle ne tient pas compte de l'état de fatigue au moment où chaque segment est couru.
>
> **2. Terrain naturel vs tapis roulant.** La variabilité du sol (rochers, boue, neige, racines), le dévers latéral et les irrégularités de surface augmentent le coût énergétique de 5 à 15 % par rapport au tapis roulant, de façon non linéaire selon le type de terrain.
>
> **3. Pentes extrêmes.** Au-delà de ±25 %, les données expérimentales de Minetti sont rares et les extrapolations moins fiables. Sur ces pentes, la marche est souvent plus économique que la course, ce que le modèle ne modélise pas explicitement.
>
> **4. Variabilité interindividuelle.** Le modèle est une courbe moyenne. L'écart entre individus sur les pentes extrêmes peut atteindre ±20 % selon la morphologie, l'expérience montagne et la technique.
>
> **En pratique**, ce plot est utile pour **identifier des tendances et des asymétries** dans ton profil de performance par rapport à une référence théorique, pas pour une mesure absolue de l'efficacité. Les écarts significatifs (> 10 %) et **reproductibles sur plusieurs courses** méritent attention. Un écart isolé sur une seule course doit être interprété avec prudence.

In [ ]:
if races and SHOW_PACE_VS_SLOPE and len(races) >= 1:
    fig = plot_pace_vs_slope_overlay(races, show_minetti=True)
    fig.show()

### Note de Greg

Je pense que je vais faire un passage spécifique sur ta course du 25/04.  
Ton cardio était hyper haut. Après, comme tu le disais, il y a la question de la fiabilité de la mesure cardio sur la montre
Chaleur ? Déshydratation ? 

In [ ]:
if races and SHOW_PACE_VS_SLOPE and len(races) >= 1:

    fig = plot_pace_vs_slope_deviation(races)
    fig.show()

---
## Section 3 — Axes d'amélioration


### 3.1 Découplage aérobie

#### 3.1.1 Qu'est-ce que le découplage aérobie ?

Quand tu cours à intensité constante, ton cœur devrait battre à une fréquence stable. En réalité, sur un effort long, la FC a tendance à **dériver progressivement vers le haut** même si la vitesse reste la même — c'est la dérive cardiaque, ou *cardiac drift*. Ce phénomène est principalement dû à la déshydratation progressive, à l'augmentation de la température corporelle, et à la fatigue musculaire qui force le recrutement de fibres supplémentaires.

Le **découplage aérobie** mesure cette dérive de façon normalisée. Plutôt que de regarder la FC seule (qui dépend de l'intensité), on suit le ratio **FC / GAP** — le coût cardiaque pour une allure donnée. Si ce ratio augmente au fil de la course, le système cardiovasculaire travaille de plus en plus dur pour maintenir la même vitesse : c'est le signal d'une fatigue aérobie croissante.



#### 3.1.2 Méthode de calcul — TrainingPeaks / Coggan

Le calcul suit la méthode de référence Pa:Hr (*Pace to Heart Rate*) définie par Andrew Coggan et popularisée par TrainingPeaks. La course est découpée en **deux moitiés égales par distance**. La première moitié sert de référence — elle représente l'état "frais" du coureur. Le découplage est l'écart relatif du ratio FC/GAP entre la deuxième moitié et la première :

$$\text{Découplage} = \frac{\text{ratio FC/GAP}_{2\text{e moitié}} - \text{ratio FC/GAP}_{1\text{re moitié}}}{\text{ratio FC/GAP}_{1\text{re moitié}}} \times 100$$

Cette définition est cohérente quelle que soit la longueur de la course — 10 km ou 160 km. Le plot continu montre l'évolution point par point du ratio normalisé, lissé sur une fenêtre glissante de 2 km.

#### 3.1.3 Comment lire le plot

La courbe violette montre l'évolution du découplage au fil de la distance. La ligne tiretée rouge marque le **seuil d'alerte à +2.5 %**. La zone rouge au-dessus du seuil matérialise visuellement les moments où le découplage devient préoccupant.

Un profil "sain" montre une courbe qui reste proche de zéro sur la première moitié, avec une légère dérive en fin de course. Un profil problématique montre une dérive précoce et soutenue — souvent le signe d'un départ trop rapide, d'une hydratation insuffisante, ou d'une chaleur excessive.

Le tableau sous le plot donne la valeur de découplage par section entre ravitaillements. C'est la lecture la plus utile pour le coach : elle permet d'identifier **quelle section précise** concentre la dégradation.



#### 3.1.4 Ce que ça apporte

Le découplage est un indicateur de **durabilité aérobie**, distinct de la performance instantanée. Deux coureurs peuvent avoir le même GAP médian sur une course mais des profils de découplage très différents — l'un dérive fortement dès le km 20, l'autre reste stable jusqu'au km 50. Le second est dans un meilleur état physiologique même s'ils arrivent au même temps.

Les axes d'amélioration dépendent du moment où le découplage apparaît :

- **Dérive précoce (avant 30 % de la course)** : le coureur est parti trop vite, ou son niveau aérobie de base est insuffisant pour l'intensité choisie. Axe de travail : travail en zone 2 prolongé, gestion du départ.
- **Dérive tardive (après 60–70 %)** : normal sur un ultra, mais si elle est très marquée (> 15–20 %), cela pointe vers une insuffisance de préparation spécifique longue distance, ou un problème de nutrition/hydratation en course. Axe de travail : sorties très longues, protocoles de nutrition à l'effort.
- **Dérive absente** : le coureur a sous-estimé ses capacités ou a couru trop prudemment. Utile à croiser avec le split ratio (section 2.2) — si le split est négatif et le découplage faible, il y a probablement une marge de progression sur le rythme de départ.

#### 3.1.5 Seuils de référence

| Découplage | Interprétation |
|---|---|
| < 2.5 % | ✅ Excellent — effort bien géré |
| 2.5–5 % | 🟡 Acceptable sur un ultra long |
| 5–10 % | 🟠 Dérive significative — à investiguer |
| > 10 % | 🔴 Dérive importante — problème de gestion ou de condition |

Ces seuils sont indicatifs et doivent être interprétés en fonction de la distance et du dénivelé de la course. Un découplage de 8 % sur un 160 km est moins préoccupant que le même découplage sur un 20 km.

*Références : Coggan A (2003). Aerobic decoupling — Pa:Hr method. TrainingPeaks. / Maunder E, Seiler S, Mildenhall MJ, Kilding AE & Plews DJ (2021). The importance of 'durability' in the physiological profiling of endurance athletes. Sports Med 51(8):1648-1651. / Smyth B, Lawlor A & Coyle R (2022). Predicting the performance-affecting factors in recreational marathon runners. Front Physiol 13:868903.*

In [ ]:
if SHOW_DECOUPLING and "heart_rate" in df.columns:
    fig = plot_aerobic_decoupling(df, RAVITO_KM, RAVITO_NOM,
                                  show_elevation=True)
    fig.show()
    dc_df = compute_aerobic_decoupling(df, RAVITO_KM, RAVITO_NOM)
    display(dc_df)

### 3.2 Variabilité d'allure (CV du GAP)

#### 3.2.1 Qu'est-ce que la variabilité d'allure ?

Sur une course trail, l'allure varie naturellement avec le relief — c'est inévitable et souhaitable. La question n'est pas de savoir si tu varies ton allure, mais **de combien et à quel moment**. Un coureur qui accélère brutalement dans les descentes et se retrouve à marcher en fin de montée gère son effort de façon erratique. Un coureur qui maintient une intensité d'effort constante (mesurée par le GAP) sur des pentes variées gère son effort de façon efficace.

Le **coefficient de variation du GAP** (CV GAP) mesure cette régularité. Il est défini comme l'écart-type du GAP divisé par sa moyenne, exprimé en pourcentage. Plus le CV est faible, plus l'allure est régulière — non pas en vitesse absolue, mais en effort relatif au dénivelé.

#### Pourquoi le GAP et pas l'allure brute ?

L'allure brute (min/km) varie mécaniquement avec la pente — même un coureur parfaitement régulier en termes d'effort aura une allure brute très variable sur un parcours montagneux. Le GAP (Grade Adjusted Pace) neutralise cette variation topographique. Mesurer la variabilité sur le GAP donne donc une image fidèle de la **gestion de l'effort**, indépendamment du profil.

#### 3.2.2 Comment lire les plots ? 

**Panel haut — boîtes à moustaches par section** : chaque boîte représente la distribution du GAP sur une section entre ravitaillements. La ligne centrale est la médiane, la boîte couvre les quartiles 25–75 %, les moustaches s'étendent jusqu'aux valeurs extrêmes. Une boîte étroite = allure régulière sur cette section. Une boîte large = allure très variable. Le chiffre au-dessus de chaque boîte est le GAP médian formaté en MM'SS"/km.

**Panel bas — barres CV par section** : le CV de chaque section, avec un code couleur :
- 🟢 Vert (CV < 15 %) : allure régulière
- 🟠 Orange (15–25 %) : allure variable
- 🔴 Rouge (> 25 %) : allure très irrégulière



#### 3.2.3 Ce que ça apporte

Un CV élevé sur une section spécifique est un signal d'investigation. Les causes possibles sont multiples : section avec profil très haché (alternance rapide montée/descente), mauvaise gestion de l'énergie à cet endroit, fatigue qui provoque des accélérations compensatoires suivies de ralentissements, ou simplement un embouteillage au départ.

La corrélation entre CV et performance est bien documentée : les coureurs qui maintiennent un CV faible sur l'ensemble de la course tendent à mieux performer, particulièrement sur les formats longs où la gestion de l'énergie est déterminante. Cela dit, un CV très faible sur une section de descente peut aussi signaler une **prudence excessive** — le coureur ne se permet pas d'exploiter son potentiel dans les descentes.

*Références : Haney TA & Mercer JA (2011). A description of variability of pacing in marathon distance running. Int J Exerc Sci 4(2):133-140. / Cuk I, Nikolaidis PT & Knechtle B (2024). Pacing strategies in ultramarathon running — a systematic review. Medicina 60(2):218.*

In [ ]:
if SHOW_VARIABILITY:
    fig = plot_pace_variability(df, RAVITO_KM, RAVITO_NOM)
    fig.show()

### 3.3 Profil circadien



#### 3.3.1 Le rythme circadien et la performance sportive

Le corps humain fonctionne selon une horloge biologique interne de 24 heures, le **rythme circadien**. Cette horloge régule la température corporelle, la sécrétion d'hormones, la vigilance, la force musculaire et la capacité aérobie. Ces paramètres ne sont pas constants au fil de la journée, ils suivent des cycles prévisibles qui influencent directement la performance physique, indépendamment de la fatigue accumulée.

Sur un ultra-trail nocturne comme la SaintéLyon (départ après 23h) ou l'UTMB (départ 18h), le coureur traverse obligatoirement le **nadir circadien** — la période de 02h à 06h du matin où les performances physiques et cognitives sont au plus bas. Cette dégradation est physiologique, non pathologique, et distincte de la fatigue musculaire ou de l'épuisement des réserves énergétiques.

#### 3.3.2 Ce que le nadir circadien implique en pratique

Pendant le nadir (barres violettes dans le plot), plusieurs phénomènes se produisent simultanément :

- La température corporelle centrale est à son minimum, ce qui réduit la vitesse de contraction musculaire
- La vigilance est au plus bas, augmentant le risque de chutes et d'erreurs de navigation
- La perception de l'effort est augmentée — le même GAP "coûte" subjectivement plus cher
- La motilité gastrique ralentit, compliquant l'absorption des glucides aux ravitaillements

Des études sur des formats ultra montrent des ralentissements de 5 à 15 % pendant cette fenêtre par rapport aux heures précédentes, même en contrôlant le profil et la fatigue.

#### 3.3.3 Comment lire le plot ?

Chaque barre représente une tranche horaire de 2 heures. La hauteur indique le GAP médian (axe inversé : vers le bas = plus rapide), la FC médiane, et le pourcentage de marche — selon les données disponibles. La ligne en tirets blancs est la médiane globale de la course.

Les barres violettes correspondent aux tranches nocturnes (22h–06h). Un ralentissement visible dans ces tranches est **attendu et normal** — il ne doit pas être interprété comme une défaillance du coureur ni conduire à corriger l'entraînement. En revanche, un ralentissement **disproportionné** pendant le nadir par rapport aux autres coureurs du même niveau peut pointer vers une mauvaise gestion du sommeil pré-course ou une sensibilité circadienne plus marquée.



#### 3.3.4 Ce que ça apporte

Ce plot est particulièrement utile pour **planifier la stratégie de course** plutôt que pour diagnostiquer un problème d'entraînement. Il permet de répondre à des questions concrètes :

- À quelle heure le coureur entre-t-il dans le nadir ? Sera-t-il dans une montée ou une descente technique à ce moment ?
- Peut-on planifier un ravitaillement "stratégique" (café, nourriture solide, contact avec l'équipe) en début de nadir pour en atténuer les effets ?
- Le coureur a-t-il une expérience préalable des courses nocturnes, ou est-ce sa première fois à 3h du matin ?

L'adaptation au travail nocturne est entraînable : les coureurs qui ont l'habitude des entraînements nocturnes ou des décalages horaires montrent une atténuation du nadir. Un bloc de préparation spécifique avec quelques sorties longues nocturnes est une stratégie validée avant un ultra nocturne.

*Références : Czeisler CA et al. (1999). Stability, precision, and near-24-hour period of the human circadian pacemaker. Science 284(5423):2177-2181. / Bearden SE & van Woerden I (2025). Circadian effects on ultra-endurance performance. PLoS ONE. / Drust B, Waterhouse J, Atkinson G, Edwards B & Reilly T (2005). Circadian rhythms in sports performance — an update. Chronobiol Int 22(1):21-44.*

In [ ]:
if SHOW_CIRCADIAN:
    fig = plot_circadian_profile(df, bin_hours=2)
    fig.show()

### 3.4 Métriques de foulée

#### 3.4.1 Cadence, longueur de foulée et fatigue neuromusculaire

La cadence (nombre de pas par minute) et la longueur de foulée sont deux variables complémentaires qui définissent la vitesse de course :

$$V = \text{Cadence} \times \text{Longueur de foulée}$$

À vitesse constante, un coureur peut donc choisir de courir avec une cadence élevée et des foulées courtes, ou une cadence basse et des foulées longues. En trail, la topographie et la fatigue perturbent cet équilibre de façon caractéristique.

La fatigue neuromusculaire se manifeste en premier lieu par une **dégradation de la longueur de foulée** — le système nerveux central réduit l'amplitude des mouvements pour protéger les muscles. La cadence peut rester stable ou augmenter légèrement en compensation, mais le produit des deux (la vitesse) diminue. Ce pattern est distinct de la fatigue métabolique (épuisement des réserves) et apparaît généralement plus tôt dans la course.



#### 3.4.2 Comment sont calculées ces métriques ?

La longueur de foulée est estimée à partir de la vitesse GPS et de la cadence :

$$\text{Longueur de foulée (m)} = \frac{V \text{ (m/s)}}{\text{Cadence (Hz)}}$$

où la cadence est convertie de spm (pas par minute) en Hz (pas par seconde). Cette estimation suppose que la cadence enregistrée est la cadence totale (pas/min), pas la cadence par jambe. Certains appareils Garmin enregistrent la cadence par jambe — dans ce cas les valeurs sont doublées automatiquement si la médiane est inférieure à 120 spm.

Le **CV de cadence** (coefficient de variation) mesure la variabilité de la cadence sur chaque section. Une cadence stable indique un recrutement neuromusculaire régulier. Une cadence très variable sur une section plate signale une instabilité technique ou une fatigue avancée.

#### 3.4.3 Comment lire le tableau ?

Chaque ligne correspond à une section entre ravitaillements. Les colonnes clés :

- **Cadence méd.** : valeur centrale sur la section — une valeur qui diminue progressivement au fil des sections signale une fatigue neuromusculaire croissante
- **CV cadence (%)** : variabilité de la cadence — un CV qui augmente en fin de course est un signal précoce de dégradation technique
- **Foulée méd. (m)** : longueur moyenne de foulée — une réduction progressive de cette valeur est le signe le plus direct de fatigue neuromusculaire


#### 3.4.4 Ce que ça apporte ?

Les métriques de foulée sont particulièrement utiles sur les formats longs où la fatigue neuromusculaire devient un facteur limitant distinct de la condition cardiovasculaire. Un coureur dont la longueur de foulée chute de 15–20 % entre la première et la dernière section a probablement atteint sa limite neuromusculaire avant sa limite aérobie — ce qui oriente les axes de travail vers le renforcement musculaire et les séances en état de fatigue plutôt que vers le volume cardiovasculaire.

À noter : ces métriques sont fiables uniquement si le capteur de cadence de la montre est fonctionnel et bien positionné. En terrain très technique (éboulis, végétation), les valeurs peuvent être parasitées par des mouvements parasites du poignet.

*Références : Nummela A, Rusko H & Mero A (1994). EMG activities and ground reaction forces during fatigued and nonfatigued conditions in sprint running. Med Sci Sports Exerc 26(5):605-609. / Morin JB et al. (2011). Mechanical factors associated with sprint running performance. Sports Biomech 10(2):100-116. / Tartaruga MP et al. (2012). The relationship between running economy and biomechanical variables. J Strength Cond Res 26(5):1338-1344.*

In [ ]:
if SHOW_STRIDE and "cadence" in df.columns:
    stride_df = compute_stride_metrics(df, RAVITO_KM, RAVITO_NOM)
    display(stride_df)

---
## Section 4 — Comparaison deux coureurs (en test)

Compare deux coureurs sur **la même course**.



In [ ]:
# Renseigner les deux blocs ci-dessous, puis exécuter les cellules.
RUNNER_A = {
    'label':      'Hase 2026 (Laura)',
    'fit_path':   '/home/gsainton/DATA/Trail/DATA/Laura/TrailRunning_2026-05-03T09_00_02_LB.fit',
    'ravito_km':  [10.4],
    'ravito_nom': ['ravito1'],
    "fc_max":     200,
    "fc_min":     40,
    "poids_kg":   52.0,
}

RUNNER_B = {
    'label':      'Haze 2026 (Greg)',
    'fit_path':   '/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/Trail-de-la-haze_2026_05_03 09:00:00.fit',
    'ravito_km':  [10.4],
    'ravito_nom': ['ravito1'],
    "fc_max":     183,
    "fc_min":     47,
    "poids_kg":   78.0,
}

In [ ]:
if SHOW_COMPARISON:

    race_a = load_and_process_race(
        fit_path   = RUNNER_A["fit_path"],
        fc_max     = RUNNER_A["fc_max"],
        fc_min     = RUNNER_A["fc_min"],
        poids_kg   = RUNNER_A["poids_kg"],
        ravito_km  = RUNNER_A["ravito_km"],
        ravito_nom = RUNNER_A["ravito_nom"],
    )
    race_a["meta"]["name"] = RUNNER_A["label"]

    race_b = load_and_process_race(
        fit_path   = RUNNER_B["fit_path"],
        fc_max     = RUNNER_B["fc_max"],
        fc_min     = RUNNER_B["fc_min"],
        poids_kg   = RUNNER_B["poids_kg"],
        ravito_km  = RUNNER_B["ravito_km"],
        ravito_nom = RUNNER_B["ravito_nom"],
    )
    race_b["meta"]["name"] = RUNNER_B["label"]
    print(f"✅ {RUNNER_A['label']} et {RUNNER_B['label']} chargés.")

### 4.1 Tableau de synthèse comparatif


In [ ]:

if SHOW_COMPARISON:
    df_cmp = build_comparison_table(race_a, race_b)
    display(df_cmp)

### 4.2 Profils normalisés, delta et radar KPI

1. Profils GAP normalisés superposés
2. Delta point par point (positif = coureur B plus lent)
3. Radar KPI comparatif sur 5 axes de performance


In [ ]:

if SHOW_COMPARISON:
    fig = plot_comparison_two_runners(race_a, race_b, col="gap_s_per_km")
    fig.show()

### 4.4 Profil altimétrique et stretch

L'axe de temps est normalisé par la durée du coureur A (référence).
Coureur B dépasse x = 1.0 s'il est plus lent.

Le **stretch** (panneau bas) est l'écart en km au même instant normalisé :
vert = A en avance, rouge = B en avance.
C'est la lecture la plus directe pour identifier **où sur la course** l'écart se creuse — si le stretch augmente fortement dans une montée, le coureur B perd du terrain en montée.


In [ ]:
if SHOW_COMPARISON:
    fig = plot_altitude_vs_time(race_a, race_b)
    fig.show()

### 4.3 Vitesse par classe de pente — comparaison


In [ ]:
if SHOW_COMPARISON:
    import plotly.graph_objects as _go

    _figs_data = []
    for _race, _color in [(race_a, "#4a9ede"), (race_b, "#e07b54")]:
        _df_r = _race["df"].copy()
        _df_r["slope_bin"] = pd.cut(_df_r["slope_pct"],
                                    bins=SLOPE_BINS, labels=SLOPE_LABELS)
        _agg = (_df_r.dropna(subset=["slope_bin", "speed_kmh"])
                .groupby("slope_bin", observed=True)["speed_kmh"]
                .agg(["mean", "std", "count"]).reset_index())
        _agg.columns = ["slope_bin", "mean", "std", "count"]
        _agg = _agg[_agg["count"] >= 10]
        _figs_data.append((_agg, _race["meta"]["name"], _color))

    _fig_cmp = _go.Figure()
    for _agg, _name, _color in _figs_data:
        _fig_cmp.add_trace(_go.Bar(
            x=_agg["slope_bin"].astype(str), y=_agg["mean"],
            error_y=dict(type="data", array=_agg["std"].values, visible=True),
            name=_name, marker_color=_color,
            hovertemplate=f"{_name}<br>Pente : %{{x}}<br>Vitesse : %{{y:.2f}} km/h<extra></extra>",
        ))
    _fig_cmp.update_layout(
        barmode="group",
        title="Vitesse par classe de pente — comparaison deux coureurs",
        xaxis_title="Classe de pente (%)",
        yaxis_title="Vitesse (km/h)",
        template="plotly_dark", height=420,
        margin=dict(l=60, r=40, t=60, b=80),
    )
    _fig_cmp.show()